# Exercício 1 — Valores Ausentes

Análise do Melbourne Housing Snapshot, preservando o conjunto original e comparando remoção, imputação estatística e KNNImputer.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

arquivo = Path("melb_data.csv")
if not arquivo.exists():
    arquivo = Path("melb_data.csv")
df = pd.read_csv(arquivo)
df.head()

## Diagnóstico inicial

In [ ]:
diagnostico = pd.DataFrame({"tipo": df.dtypes.astype(str), "ausentes": df.isna().sum(), "percentual_ausentes": df.isna().mean().mul(100), "distintos": df.nunique(dropna=True)})
print("Dimensões:", df.shape)
diagnostico

## Resumo de valores ausentes

In [ ]:
resumo_ausentes = (df.isna().sum().rename("quantidade").to_frame().assign(percentual=lambda x: x["quantidade"].div(len(df)).mul(100)).query("quantidade > 0").sort_values("percentual", ascending=False))
resumo_ausentes

## Visualização do padrão de ausência

In [ ]:
try:
    import missingno as msno
    msno.matrix(df, figsize=(12, 5))
    plt.show()
except ImportError:
    sns.heatmap(df.isna(), cbar=False, yticklabels=False)
    plt.show()

plt.figure(figsize=(10, 4))
sns.barplot(x=resumo_ausentes.index, y=resumo_ausentes["percentual"], color="#4472c4")
plt.xticks(rotation=70)
plt.ylabel("Percentual ausente")
plt.tight_layout()
plt.show()

## Estratégia 1 — Remoção

A remoção foi aplicada às colunas com mais de 40% de ausência e, depois, às linhas que ainda possuem ausências. A escolha evita que uma coluna com pouca informação seja preenchida artificialmente e permite comparar o impacto de uma base completa.

In [ ]:
df_remocao = df.copy()
colunas_removidas = resumo_ausentes.query("percentual > 40").index.tolist()
df_remocao = df_remocao.drop(columns=colunas_removidas).dropna()
colunas_removidas, df_remocao.shape

## Estratégia 2 — Imputação

In [ ]:
df_imputacao = df.copy()
numericas = df_imputacao.select_dtypes(include=np.number).columns
categoricas = df_imputacao.select_dtypes(exclude=np.number).columns
for coluna in numericas:
    df_imputacao[coluna] = df_imputacao[coluna].fillna(df_imputacao[coluna].median())
for coluna in categoricas:
    moda = df_imputacao[coluna].mode(dropna=True)
    if not moda.empty:
        df_imputacao[coluna] = df_imputacao[coluna].fillna(moda.iloc[0])
df_imputacao.shape, int(df_imputacao.isna().sum().sum())

## Comparação entre estratégias

In [ ]:
colunas_tratadas = resumo_ausentes.index.intersection(df.select_dtypes(include=np.number).columns)
comparacao = pd.DataFrame({
    "original_linhas": [len(df)],
    "remocao_linhas": [len(df_remocao)],
    "imputacao_linhas": [len(df_imputacao)],
    "ausentes_original": [int(df.isna().sum().sum())],
    "ausentes_remocao": [int(df_remocao.isna().sum().sum())],
    "ausentes_imputacao": [int(df_imputacao.isna().sum().sum())]
})
estatisticas = pd.concat({"original": df[colunas_tratadas].agg(["mean", "median"]), "remocao": df_remocao[colunas_tratadas].agg(["mean", "median"]), "imputacao": df_imputacao[colunas_tratadas].agg(["mean", "median"])}, axis=1)
comparacao, estatisticas

## Desafio — KNNImputer

In [ ]:
from sklearn.impute import KNNImputer
base_knn = df.select_dtypes(include=np.number).copy()
imputer = KNNImputer(n_neighbors=5)
resultado_knn = pd.DataFrame(imputer.fit_transform(base_knn), columns=base_knn.columns, index=base_knn.index)
comparacao_knn = pd.DataFrame({"mediana": df[base_knn.columns].median(), "imputacao_knn": resultado_knn.loc[df[base_knn.columns].isna().any(axis=1)].median()})
comparacao_knn

## Respostas de reflexão

Uma coluna com mais da metade dos valores ausentes não deve ser automaticamente imputada ou descartada. A decisão depende da importância da variável, do mecanismo de ausência, da existência de variáveis relacionadas e do objetivo da análise. Se houver pouca informação e a imputação criar muitos valores artificiais, a remoção tende a ser mais adequada; se a coluna for essencial, pode ser preferível manter uma indicação de ausência e usar uma imputação cuidadosa.

Em variáveis como Price e Landsize, a mediana costuma ser mais segura quando a distribuição é assimétrica ou contém imóveis muito caros e terrenos muito grandes. A média pode ser influenciada por esses extremos.

A ausência pode ser informativa quando representa uma decisão do processo de registro, como um imóvel sem segundo banheiro porque a característica não se aplica, ou quando determinados anúncios deixam campos sem preenchimento de forma sistemática.

O KNNImputer usa a proximidade entre observações e pode produzir valores mais específicos do que a mediana, mas exige mais processamento, depende da escala das variáveis e pode incorporar relações que não representam causalidade.